# 07A — Augmented Granger Causality Tests (1985–2020)

**Purpose:** Re-run Granger causality and panel logistic tests on the augmented dataset.

**Key differences vs NB07:**
- Macro data now covers 1985–2020 (vs 1997–2020 in NB07)
- Granger tests for macro variables (credit growth → crisis) run on longer series
- Sentiment Granger tests remain restricted to 2003–2020 (where BIS speeches exist)
- Panel logistic tests run separately on Stage 1 (macro, 1988–2020) and Stage 2 (sentiment, 2003–2020)

**Inputs:**
- `data/processed/augmented_analysis/df_macro_augmented.csv` — Stage 1 dataset
- `data/processed/augmented_analysis/df_sent_augmented.csv` — Stage 2 dataset

---

## Cell 1 — Imports and paths

In [3]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from statsmodels.tsa.stattools import grangercausalitytests
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant
from scipy import stats

BASE    = Path(r'C:\Users\Owner\OneDrive\dissertation')
AUG_DIR = BASE / 'data' / 'processed' / 'augmented_analysis'
OUT_DIR = BASE / 'data' / 'processed'
FIG_DIR = BASE / 'figures' / 'augmented_analysis'
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Stage 1: macro panel (1988–2020)
df_macro = pd.read_csv(AUG_DIR / 'df_macro_augmented.csv')
df_macro = df_macro.sort_values(['iso','year']).reset_index(drop=True)

# Stage 2: sentiment panel (2003–2020)
df_sent = pd.read_csv(AUG_DIR / 'df_sent_augmented.csv')
df_sent = df_sent.sort_values(['iso','year']).reset_index(drop=True)

TARGET = next(c for c in df_macro.columns if 'target' in c.lower())
SENT_COL = 'P_neg'  # primary sentiment measure

print(f'Stage 1 macro panel : {df_macro.shape}  ({df_macro["year"].min()}–{df_macro["year"].max()})')
print(f'Stage 2 sent panel  : {df_sent.shape}   ({df_sent["year"].min()}–{df_sent["year"].max()})')
print(f'Target              : {TARGET}')

req_sent = [c for c in ['P_neg','P_pos','net_sentiment'] if c in df_sent.columns]
print(f'Sentiment measures  : {req_sent}')

Stage 1 macro panel : (569, 101)  (1989–2020)
Stage 2 sent panel  : (319, 141)   (2003–2020)
Target              : target_h1
Sentiment measures  : []


## Cell 2 — Credit growth feature (Stage 1: full extended panel)

In [5]:
df_macro['tloans_gr'] = df_macro.groupby('iso')['tloans'].pct_change()
df_sent['tloans_gr']  = df_sent.groupby('iso')['tloans'].pct_change()

SENT_MEASURES = {
    'P_neg':         'FinBERT negativity score',
    'P_pos':         'FinBERT positivity score',
    'net_sentiment': 'FinBERT net sentiment (P_pos - P_neg)',
}

print('Sentiment measures available in Stage 2 dataset:')
for k, v in SENT_MEASURES.items():
    present = k in df_sent.columns
    print(f'  {k}: {v}  ({"present" if present else "MISSING"})') 
print()
for col in ['tloans_gr', TARGET]:
    n_null_m = df_macro[col].isna().sum() if col in df_macro.columns else 'N/A'
    print(f'Stage 1 {col}: NaN={n_null_m}')

Sentiment measures available in Stage 2 dataset:
  P_neg: FinBERT negativity score  (MISSING)
  P_pos: FinBERT positivity score  (MISSING)
  net_sentiment: FinBERT net sentiment (P_pos - P_neg)  (MISSING)

Stage 1 tloans_gr: NaN=19
Stage 1 target_h1: NaN=0


## Cell 3 — Granger tests: credit growth → crisis (Stage 1, extended panel)
This test now uses the longer 1988–2020 series, providing more statistical power.

In [7]:
from statsmodels.tsa.stattools import grangercausalitytests

print('=== GRANGER TEST: Credit growth → Crisis (Stage 1: 1988–2020) ===')
print('Panel approach: test per country, report country-level p-values')
print()

results_granger = []
MAX_LAG = 3

for ctry in sorted(df_macro['iso'].unique()):
    sub = df_macro[df_macro['iso'] == ctry][['tloans_gr', TARGET]].dropna()
    if len(sub) < 10:
        print(f'  {ctry}: insufficient observations ({len(sub)}) — skip')
        continue
    try:
        gc = grangercausalitytests(sub, maxlag=MAX_LAG, verbose=False)
        for lag in range(1, MAX_LAG+1):
            pval = gc[lag][0]['ssr_ftest'][1]
            results_granger.append({'country': ctry, 'lag': lag, 'p_value': pval,
                                     'significant_05': pval < 0.05})
    except Exception as e:
        print(f'  {ctry}: {e}')

gc_df = pd.DataFrame(results_granger)

print(f'{"Country":<6} {"Lag1 p":>10} {"Lag2 p":>10} {"Lag3 p":>10}')
print('-' * 40)
for ctry in sorted(gc_df["country"].unique()):
    sub = gc_df[gc_df['country']==ctry].sort_values('lag')
    pvals = [f'{row["p_value"]:.4f}{"*" if row["p_value"]<0.05 else ""}'
             for _, row in sub.iterrows()]
    while len(pvals) < 3: pvals.append('  n/a')
    print(f'{ctry:<6} {pvals[0]:>10} {pvals[1]:>10} {pvals[2]:>10}')

print()
n_sig = (gc_df['p_value'] < 0.05).sum()
print(f'Significant at 5%: {n_sig} of {len(gc_df)} country-lag tests')
gc_df.to_csv(OUT_DIR / 'augmented_granger_credit_results.csv', index=False)
print(f'Saved augmented_granger_credit_results.csv')

=== GRANGER TEST: Credit growth → Crisis (Stage 1: 1988–2020) ===
Panel approach: test per country, report country-level p-values

  AUS: The x values include a column with constant values and so the test statistic cannot be computed.
  CAN: The x values include a column with constant values and so the test statistic cannot be computed.
  FIN: The x values include a column with constant values and so the test statistic cannot be computed.
  NOR: The x values include a column with constant values and so the test statistic cannot be computed.
Country     Lag1 p     Lag2 p     Lag3 p
----------------------------------------
BEL       0.0093*    0.0108*    0.0409*
CHE        0.0812     0.3396     0.3698
DEU        0.6663     0.2238     0.4720
DNK        0.1665    0.0242*     0.0925
ESP       0.0491*     0.3416     0.3753
FRA        0.5580    0.0122*    0.0218*
GBR        0.3433     0.2617     0.3440
IRL        0.0759     0.2118     0.0559
ITA        0.5808     0.2355     0.1648
JPN        

## Cell 4 — Granger tests: sentiment → crisis (Stage 2, 2003–2020)
Sentiment tests unchanged — restricted to 2003–2020 as in NB07.

In [9]:
print('=== GRANGER TEST: Sentiment → Crisis (Stage 2: 2003–2020) ===')
print('Identical to NB07 — using Stage 2 dataset for consistency')
print()

TARGET_SENT = next(c for c in df_sent.columns if 'target' in c.lower())
sent_gc_results = []

for sent_col in [c for c in SENT_MEASURES if c in df_sent.columns]:
    print(f'Testing: {sent_col} → {TARGET_SENT}')
    for ctry in sorted(df_sent['iso'].unique()):
        sub = df_sent[df_sent['iso']==ctry][[sent_col, TARGET_SENT]].dropna()
        if len(sub) < 8: continue
        try:
            gc = grangercausalitytests(sub, maxlag=2, verbose=False)
            for lag in [1, 2]:
                pval = gc[lag][0]['ssr_ftest'][1]
                sent_gc_results.append({
                    'sentiment': sent_col, 'country': ctry,
                    'lag': lag, 'p_value': pval
                })
        except Exception as e:
            pass

sgc_df = pd.DataFrame(sent_gc_results)
if len(sgc_df) > 0:
    print()
    for sm in sgc_df['sentiment'].unique():
        sub = sgc_df[sgc_df['sentiment']==sm]
        n_sig = (sub['p_value'] < 0.05).sum()
        print(f'  {sm}: {n_sig}/{len(sub)} country-lag tests significant at 5%')
    sgc_df.to_csv(OUT_DIR / 'augmented_granger_sentiment_results.csv', index=False)
    print('\nSaved augmented_granger_sentiment_results.csv')
else:
    print('No results — check sentiment columns in df_sent')

=== GRANGER TEST: Sentiment → Crisis (Stage 2: 2003–2020) ===
Identical to NB07 — using Stage 2 dataset for consistency

No results — check sentiment columns in df_sent


## Cell 5 — Panel logistic: Stage 1 (macro controls, extended period)

In [11]:
print('=== PANEL LOGISTIC — STAGE 1: macro controls (1988–2020) ===')
print('Increased observations allow more stable coefficient estimates.')
print()

# Use demeaned features already computed in NB05A
macro_control_cols = [c for c in df_macro.columns
                      if '_lag1_dm' in c and 'tloans' in c]

if len(macro_control_cols) > 0:
    ctrl = macro_control_cols[:3]  # use up to 3 controls to avoid collinearity
    reg_df = df_macro[ctrl + [TARGET]].dropna()
    X = add_constant(reg_df[ctrl])
    y = reg_df[TARGET]

    from statsmodels.discrete.discrete_model import Logit
    try:
        logit_m = Logit(y, X).fit(disp=0)
        print(logit_m.summary2())
        print(f'\nPseudo-R²: {logit_m.prsquared:.4f}')
    except Exception as e:
        print(f'Logit fitting error: {e}')
        print('Falling back to OLS for illustrative coefficient direction:')
        ols_m = OLS(y, X).fit()
        print(ols_m.summary2())
else:
    print('Demeaned macro columns not found in df_macro — run NB05A first.')

=== PANEL LOGISTIC — STAGE 1: macro controls (1988–2020) ===
Increased observations allow more stable coefficient estimates.

                                Results: Logit
Model:                    Logit                Method:               MLE      
Dependent Variable:       target_h1            Pseudo R-squared:     0.080    
Date:                     2026-04-09 05:30     AIC:                  161.1484 
No. Observations:         569                  BIC:                  178.5239 
Df Model:                 3                    Log-Likelihood:       -76.574  
Df Residuals:             565                  LL-Null:              -83.269  
Converged:                1.0000               LLR p-value:          0.0038670
No. Iterations:           8.0000               Scale:                1.0000   
------------------------------------------------------------------------------
                              Coef.  Std.Err.    z     P>|z|   [0.025   0.975]
-------------------------------------

## Cell 6 — Panel logistic: Stage 2 (sentiment incremental, 2003–2020)

In [13]:
print('=== PANEL LOGISTIC — STAGE 2: sentiment increment (2003–2020) ===')
print('Pseudo-R² increment from adding sentiment over macro controls.')
print()

from statsmodels.discrete.discrete_model import Logit

TARGET_SENT = next(c for c in df_sent.columns if 'target' in c.lower())

macro_cols_sent = [c for c in df_sent.columns if '_lag1_dm' in c and 'tloans' in c]
sent_rdm_cols   = [c for c in df_sent.columns if '_lag1_rdm' in c and 'P_neg' in c]

if len(macro_cols_sent) == 0:
    print('No demeaned macro columns in df_sent — check NB05A outputs.')
else:
    ctrl = macro_cols_sent[:2]
    reg_df = df_sent[ctrl + sent_rdm_cols[:2] + [TARGET_SENT]].dropna()

    # Macro-only model
    X_macro = add_constant(reg_df[ctrl])
    try:
        m_macro = Logit(reg_df[TARGET_SENT], X_macro).fit(disp=0)
        pr2_macro = m_macro.prsquared
        print(f'Macro-only pseudo-R²  : {pr2_macro:.4f}')
    except Exception as e:
        pr2_macro = None
        print(f'Macro-only fit failed: {e}')

    # Macro + sentiment model
    if len(sent_rdm_cols) > 0:
        X_sent = add_constant(reg_df[ctrl + sent_rdm_cols[:2]])
        try:
            m_sent = Logit(reg_df[TARGET_SENT], X_sent).fit(disp=0)
            pr2_sent = m_sent.prsquared
            print(f'Macro+sent pseudo-R²  : {pr2_sent:.4f}')
            if pr2_macro:
                print(f'Increment from sentiment: {pr2_sent - pr2_macro:+.4f}')
            print()
            # Print sentiment coefficient
            for col in sent_rdm_cols[:2]:
                if col in m_sent.params.index:
                    pval = m_sent.pvalues[col]
                    coef = m_sent.params[col]
                    print(f'  {col}: coef={coef:.4f}  p={pval:.4f}  {"*" if pval<0.05 else "(n.s.)"}')          
        except Exception as e:
            print(f'Macro+sent fit failed: {e}')
    else:
        print('No rolling-demeaned sentiment columns found in df_sent.')

print()
print('NOTE: With 26 crisis events, logistic coefficients have low precision.')
print('      Interpret as directional indicators, not precise estimates.')

=== PANEL LOGISTIC — STAGE 2: sentiment increment (2003–2020) ===
Pseudo-R² increment from adding sentiment over macro controls.

Macro-only pseudo-R²  : 0.1187
Macro+sent pseudo-R²  : 0.1531
Increment from sentiment: +0.0344

  P_neg_lag1_rdm: coef=-6.9084  p=0.1026  (n.s.)
  P_neg_change_lag1_rdm: coef=-0.8736  p=0.8221  (n.s.)

NOTE: With 26 crisis events, logistic coefficients have low precision.
      Interpret as directional indicators, not precise estimates.


In [14]:
print('=' * 65)
print(' NB07A AUGMENTED GRANGER CAUSALITY COMPLETE')
print('=' * 65)
print()
print('Next: Run NB08A_Augmented_Robustness_Checks.ipynb')

 NB07A AUGMENTED GRANGER CAUSALITY COMPLETE

Next: Run NB08A_Augmented_Robustness_Checks.ipynb
